# Anexo I: Web Scrapping

In [ ]:
#Instalar librerías
!pip install requests beautifulsoup4

In [ ]:
# Importar librerías
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [ ]:
#petición a la web
url = "https://www.autohero.com/es/search/"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)
response.status_code

200

In [ ]:
# Parsear el HTML
soup = BeautifulSoup(response.text, "html.parser")

In [ ]:
import requests
import pandas as pd

url = "https://www.autohero.com/es/v1/search/"

headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json"
}

In [ ]:
params = {
    "page": 1,
    "size": 50,
    "sort": "recommended"
}

In [ ]:
response = requests.get(url, headers=headers, params=params)

response.status_code

200

In [ ]:
response.text[:500]

'<!doctype html>\n<html lang="es">\n  <head>\n    <link rel="icon" sizes="16x16" href="https://artifacts-cdn.autohero.com/master/icons/icon_16x16.png" type="image/png"></link><link rel="icon" sizes="32x32" href="https://artifacts-cdn.autohero.com/master/icons/icon_32x32.png" type="image/png"></link><link rel="apple-touch-icon" sizes="180x180" href="https://artifacts-cdn.autohero.com/master/icons/icon_180x180.png" type="undefined"></link>\n    <link rel="dns-prefetch" href="//cdn.mouseflow.com" /><lin'

## Extracción de datos: web scrapping de la web de Autohero

Se procede al scrapeo de datos de Autohero para obtener un nuevo Dataset y poder realizar la integración de datos: combinando los Datasets y generando un DataFrame final coherente

In [ ]:
import requests
import time
import pandas as pd

URL = "https://api-customer.prod.retail.auto1.cloud/v1/retail-customer-gateway/graphql"

HEADERS = {
    "Content-Type": "application/json",
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.0.0 Safari/537.36",
}

LIMIT = 24

def fetch_data(offset):
    data = {
        "operationName": "searchAdV9AdsV2",
        "variables": {
            "search": {
                "offset": offset,
                "limit": LIMIT,
                "sort": "most_popular",
                "filter": {
                    "field": "countryCode",
                    "op": "eq",
                    "value": "ES"
                },
                "aggs": [],
                "fields": [
                    "registration",
                    "firstEligibleToBePurchasedAt",
                    "variablePositioning",
                    "doorCount",
                    "seatCount",
                    "bodyType",
                    "outerColor",
                    "outerColorType",
                    "upholstery"
                ],
                "postFilter": None,
                "properties": {
                    "abTestViewParams": None,
                    "firstPublishedDays": -365,
                    "shuffleCategoryBResults": True
                }
            },
            "tradeInId": None
        },
        "query": """
        query searchAdV9AdsV2($search: EsSearchRequestProjectionInput!, $tradeInId: UUID) {
            searchAdV9AdsV2(search: $search, tradeInId: $tradeInId)
        }
        """
    }

    try:
        response = requests.post(URL, headers=HEADERS, json=data, timeout=15)
        response.raise_for_status()
        return response.json()
    except requests.RequestException as e:
        print(f"Error en la petición offset {offset}: {e}")
        return None
def main():
    total_coches = []
    offset = 0
    total = None

    while True:
        print(f"Solicitando offset {offset}...")
        data = fetch_data(offset)

        if data is None:
            print("No hay datos, reintentando en 5 segundos...")
            time.sleep(5)
            continue

        parsear = data.get("data", {}).get("searchAdV9AdsV2")
        if not parsear:
            print("No se encontró el campo 'searchAdV9AdsV2', terminando.")
            break

        if total is None:
            total = parsear.get("total", 0)
            print(f"Total coches estimados: {total}")

        coches = parsear.get("data", [])
        if not coches:
            print("No hay más coches, terminando.")
            break

        total_coches.extend(coches)
        print(f"Coches recolectados: {len(total_coches)} / {total}")

        if len(total_coches) >= total:
            break

        offset += LIMIT
        time.sleep(1)

    print(f"Descarga completa. Total coches: {len(total_coches)}")

    df = pd.DataFrame(total_coches)
    df.to_csv("coches_autohero.csv", index=False)
    print("Datos guardados en 'coches_autohero.csv'")


if __name__ == "__main__":
    main()

Solicitando offset 0...
Total coches estimados: 4635
Coches recolectados: 24 / 4635
Solicitando offset 24...
Coches recolectados: 48 / 4635
Solicitando offset 48...
Coches recolectados: 72 / 4635
Solicitando offset 72...
Coches recolectados: 96 / 4635
Solicitando offset 96...
Coches recolectados: 120 / 4635
Solicitando offset 120...
Coches recolectados: 144 / 4635
Solicitando offset 144...
Coches recolectados: 168 / 4635
Solicitando offset 168...
Coches recolectados: 192 / 4635
Solicitando offset 192...
Coches recolectados: 216 / 4635
Solicitando offset 216...
Coches recolectados: 240 / 4635
Solicitando offset 240...
Coches recolectados: 264 / 4635
Solicitando offset 264...
Coches recolectados: 288 / 4635
Solicitando offset 288...
Coches recolectados: 312 / 4635
Solicitando offset 312...
Coches recolectados: 336 / 4635
Solicitando offset 336...
Coches recolectados: 360 / 4635
Solicitando offset 360...
Coches recolectados: 384 / 4635
Solicitando offset 384...
Coches recolectados: 408 / 